In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Data Reading**

In [0]:
df=spark.read.format('delta')\
    .load('abfss://bronze@adventwork.dfs.core.windows.net/territories')

### **Drop Rescued Data Column**

In [0]:
df=df.drop("_rescued_data")

### **Checking Schema**

In [0]:
df.printSchema()

root
 |-- SalesTerritoryKey: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Continent: string (nullable = true)



### **Changing SalesTerritoryKey column's format**

In [0]:
df=df.withColumn('SalesTerritoryKey',col('SalesTerritoryKey').cast('int'))

### **Drop Duplicates**

In [0]:
df=df.dropDuplicates(['Country','Region','Continent'])

### **Fixing string column's data quality**

In [0]:
df=df.withColumn('Region',trim(col("Region")))\
     .withColumn('Country',trim(col("Country")))\
     .withColumn('Continent',trim(col("Continent")))

### **Incremental Loading**

In [0]:
if spark.catalog.tableExists('adventure_works.silver.territories'):
    df_silver_territories = spark.read.table('adventure_works.silver.territories')
    df=df.join(df_silver_territories,on = ['SalesTerritoryKey'],how="left_anti")

### **Data Writing**

In [0]:
if spark.catalog.tableExists('adventure_works.silver.territories'):
    df_silver_territory = spark.read.table('adventure_works.silver.territories')
    df=df.join(df_silver_territory,'SalesTerritoryKey','left_anti')


In [0]:
df.write.format('delta').mode('append')\
    .save('abfss://silver@adventwork.dfs.core.windows.net/territories')

In [0]:
%sql
create table if not exists adventure_works.silver.territories
using delta
location 'abfss://silver@adventwork.dfs.core.windows.net/territories'

In [0]:
df.display()

SalesTerritoryKey,Region,Country,Continent


In [0]:
%sql
select * from adventure_works.silver.territories

SalesTerritoryKey,Region,Country,Continent
7,France,France,Europe
4,Southwest,United States,North America
6,Canada,Canada,North America
9,Australia,Australia,Pacific
1,Northwest,United States,North America
3,Central,United States,North America
10,United Kingdom,United Kingdom,Europe
5,Southeast,United States,North America
8,Germany,Germany,Europe
2,Northeast,United States,North America
